# Feature Engineering


## Objective

This notebook builds the first baseline modeling dataset from the raw training files.

The goals are to:

- Load raw weekly training data
- Restrict the dataset to target-player rows
- Create baseline player and contextual features
- Clean selected raw variables
- Assemble a reusable modeling table
- Save the processed dataset for training


    


In [1]:
import sys
from pathlib import Path
import json
from datetime import datetime

import pandas as pd
import numpy as np

sys.path.append(str(Path("..").resolve()))

from src.data_loader import list_train_files, load_train_week
from src.features import add_basic_features, select_target_rows, select_model_columns
from src.config import PROCESSED_DIR

pd.set_option("display.max_columns", None)

## Load Raw Training Data

We begin by loading a subset of weekly training files.

For this first baseline, we use a limited number of weeks to validate the feature engineering pipeline before scaling to the full training set.

In [2]:
train_files = list_train_files()

print("Number of training files found:", len(train_files))
print("First training files:")
for f in train_files[:5]:
    print("-", f.name)

Number of training files found: 18
First training files:
- input_2023_w01.csv
- input_2023_w02.csv
- input_2023_w03.csv
- input_2023_w04.csv
- input_2023_w05.csv


In [3]:
n_weeks = 2
selected_files = train_files[:n_weeks]

print("Files selected for this baseline run:")
for f in selected_files:
    print("-", f.name)

Files selected for this baseline run:
- input_2023_w01.csv
- input_2023_w02.csv


In [4]:
dfs = [load_train_week(f) for f in selected_files]
df = pd.concat(dfs, ignore_index=True)

print("Loaded raw dataset shape:", df.shape)
display(df.head())

Loaded raw dataset shape: (574300, 23)


,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
0,2023090700,101,False,54527,1,right,42,Bryan Cook,6-1,210,1999-09-07,FS,Defense,Defensive Coverage,52.33,36.94,0.09,0.39,322.40,238.24,21,63.259998,-0.22
1,2023090700,101,False,54527,2,right,42,Bryan Cook,6-1,210,1999-09-07,FS,Defense,Defensive Coverage,52.33,36.94,0.04,0.61,200.89,236.05,21,63.259998,-0.22
2,2023090700,101,False,54527,3,right,42,Bryan Cook,6-1,210,1999-09-07,FS,Defense,Defensive Coverage,52.33,36.93,0.12,0.73,147.55,240.60,21,63.259998,-0.22
3,2023090700,101,False,54527,4,right,42,Bryan Cook,6-1,210,1999-09-07,FS,Defense,Defensive Coverage,52.35,36.92,0.23,0.81,131.40,244.25,21,63.259998,-0.22
4,2023090700,101,False,54527,5,right,42,Bryan Cook,6-1,210,1999-09-07,FS,Defense,Defensive Coverage,52.37,36.90,0.35,0.82,123.26,244.25,21,63.259998,-0.22


##  Raw Data Overview

Before filtering rows or creating new features, we briefly inspect the structure of the loaded training subset.

This helps confirm that the raw dataset has been loaded correctly and that the columns needed for feature engineering are present.

In [5]:
print("Number of columns:", df.shape[1])
print("\nColumns:")
print(df.columns.tolist())

Number of columns: 23

Columns:
['game_id', 'play_id', 'player_to_predict', 'nfl_id', 'frame_id', 'play_direction', 'absolute_yardline_number', 'player_name', 'player_height', 'player_weight', 'player_birth_date', 'player_position', 'player_side', 'player_role', 'x', 'y', 's', 'a', 'dir', 'o', 'num_frames_output', 'ball_land_x', 'ball_land_y']


In [6]:
print("Raw dataset dtypes:")
display(df.dtypes)

Raw dataset dtypes:


game_id                       int64
play_id                       int64
player_to_predict              bool
nfl_id                        int64
frame_id                      int64
play_direction               object
absolute_yardline_number      int64
player_name                  object
player_height                object
player_weight                 int64
player_birth_date            object
player_position              object
player_side                  object
player_role                  object
x                           float64
y                           float64
s                           float64
a                           float64
dir                         float64
o                           float64
num_frames_output             int64
ball_land_x                 float64
ball_land_y                 float64
dtype: object

##  Select Target Rows

The raw tracking files contain both target-player rows and contextual rows from surrounding players.

For this first baseline, we restrict the dataset to rows where:

- `player_to_predict = True`

This creates a direct supervised-learning table focused on the primary prediction targets.

In [7]:
target_df = select_target_rows(df)

print("Target dataset shape:", target_df.shape)
print("Target proportion:", len(target_df) / len(df))

display(target_df.head())

Target dataset shape: (152305, 23)
Target proportion: 0.2652011144001393


,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y
26,2023090700,101,True,46137,1,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.32,20.69,0.31,0.49,79.43,267.68,21,63.259998,-0.22
27,2023090700,101,True,46137,2,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.35,20.66,0.36,0.74,118.07,268.66,21,63.259998,-0.22
28,2023090700,101,True,46137,3,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.39,20.63,0.44,0.76,130.89,269.78,21,63.259998,-0.22
29,2023090700,101,True,46137,4,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.43,20.61,0.48,0.62,134.50,269.78,21,63.259998,-0.22
30,2023090700,101,True,46137,5,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.48,20.58,0.54,0.44,129.79,269.06,21,63.259998,-0.22


## Create Basic Features

We generate a first set of baseline engineered features, including:

- player height converted to inches,
- player age derived from birth year,
- binary play-direction encoding.

These variables provide a simple but useful first feature set for tabular modeling.

In [8]:
feat_df = add_basic_features(target_df)

print("Feature-engineered dataset shape:", feat_df.shape)
display(feat_df.head())

Feature-engineered dataset shape: (152305, 26)


,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,player_birth_date,player_position,player_side,player_role,x,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y,player_height_inches,player_age,is_moving_right
26,2023090700,101,True,46137,1,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.32,20.69,0.31,0.49,79.43,267.68,21,63.259998,-0.22,73,26,1
27,2023090700,101,True,46137,2,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.35,20.66,0.36,0.74,118.07,268.66,21,63.259998,-0.22,73,26,1
28,2023090700,101,True,46137,3,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.39,20.63,0.44,0.76,130.89,269.78,21,63.259998,-0.22,73,26,1
29,2023090700,101,True,46137,4,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.43,20.61,0.48,0.62,134.50,269.78,21,63.259998,-0.22,73,26,1
30,2023090700,101,True,46137,5,right,42,Justin Reid,6-1,204,1997-02-15,SS,Defense,Defensive Coverage,51.48,20.58,0.54,0.44,129.79,269.06,21,63.259998,-0.22,73,26,1


In [9]:
engineered_cols = ["player_height_inches", "player_age", "is_moving_right"]

print("Engineered columns present:")
print([col for col in engineered_cols if col in feat_df.columns])

Engineered columns present:
['player_height_inches', 'player_age', 'is_moving_right']


In [10]:
display(
    feat_df[
        [
            "player_height",
            "player_height_inches",
            "player_birth_date",
            "player_age",
            "play_direction",
            "is_moving_right",
        ]
    ].head()
)

,player_height,player_height_inches,player_birth_date,player_age,play_direction,is_moving_right
26,6-1,73,1997-02-15,26,right,1
27,6-1,73,1997-02-15,26,right,1
28,6-1,73,1997-02-15,26,right,1
29,6-1,73,1997-02-15,26,right,1
30,6-1,73,1997-02-15,26,right,1


## Select Modeling Columns

After engineering the baseline features, we select the columns that will be carried into the first modeling dataset.

This includes:

- tracking variables,
- player attributes,
- contextual variables,
- and target columns.

In [11]:
model_df = select_model_columns(feat_df)

print("Model dataset shape:", model_df.shape)
print("Number of columns:", model_df.shape[1])

display(model_df.head())

Model dataset shape: (152305, 21)
Number of columns: 21


,game_id,play_id,nfl_id,frame_id,x,y,s,a,dir,o,absolute_yardline_number,player_weight,player_height_inches,player_age,is_moving_right,player_position,player_side,player_role,ball_land_x,ball_land_y,num_frames_output
26,2023090700,101,46137,1,51.32,20.69,0.31,0.49,79.43,267.68,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
27,2023090700,101,46137,2,51.35,20.66,0.36,0.74,118.07,268.66,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
28,2023090700,101,46137,3,51.39,20.63,0.44,0.76,130.89,269.78,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
29,2023090700,101,46137,4,51.43,20.61,0.48,0.62,134.50,269.78,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
30,2023090700,101,46137,5,51.48,20.58,0.54,0.44,129.79,269.06,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21


In [12]:
print("Model dataset dtypes:")
display(model_df.dtypes)

Model dataset dtypes:


game_id                       int64
play_id                       int64
nfl_id                        int64
frame_id                      int64
x                           float64
y                           float64
s                           float64
a                           float64
dir                         float64
o                           float64
absolute_yardline_number      int64
player_weight                 int64
player_height_inches          int64
player_age                    int64
is_moving_right               int32
player_position              object
player_side                  object
player_role                  object
ball_land_x                 float64
ball_land_y                 float64
num_frames_output             int64
dtype: object

##  Missing Value Check

After filtering and transforming the data, we verify missing values in the engineered dataset.

This is important because feature transformations can introduce missing values even if the raw dataset was mostly complete.

In [13]:
missing_percent = model_df.isnull().mean().sort_values(ascending=False)
display(missing_percent[missing_percent > 0])

Series([], dtype: float64)

##  Target Variable Sanity Check

Before saving the modeling dataset, we inspect the target-related columns to confirm that they are present and behave as expected..

In [14]:
target_summary = model_df[["ball_land_x", "ball_land_y", "num_frames_output"]].describe()
display(target_summary)

,ball_land_x,ball_land_y,num_frames_output
count,152305.000000,152305.000000,152305.000000
mean,61.394456,26.360714,12.236105
std,26.354903,15.016233,6.292270
min,-0.380000,-1.740000,5.000000
25%,42.389999,14.140000,8.000000
50%,59.959999,25.660000,10.000000
75%,79.910004,38.650002,14.000000
max,119.779999,57.330002,94.000000


In [15]:
print("Rows with missing target values:")
print(model_df[["ball_land_x", "ball_land_y"]].isnull().sum())

Rows with missing target values:
ball_land_x    0
ball_land_y    0
dtype: int64


## Inspect Engineered Features

We briefly inspect the distributions of the newly created engineered features to verify that the transformations are reasonable.

In [16]:
display(model_df[["player_height_inches", "player_age"]].describe())

,player_height_inches,player_age
count,152305.000000,152305.000000
mean,72.862762,26.251285
std,2.161814,2.819886
min,66.000000,21.000000
25%,71.000000,24.000000
50%,73.000000,26.000000
75%,74.000000,28.000000
max,80.000000,35.000000


##  Save Processed Dataset

We save the processed baseline dataset to the `data/processed/` directory so it can be reused consistently in model training and evaluation notebooks.

In [17]:
output_path = PROCESSED_DIR / "baseline_train_v1.parquet"
model_df.to_parquet(output_path, index=False)

print("Saved processed dataset to:")
print(output_path)

Saved processed dataset to:
C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\data\processed\baseline_train_v1.parquet


In [18]:
check_df = pd.read_parquet(output_path)

print("Reloaded dataset shape:", check_df.shape)
display(check_df.head())

Reloaded dataset shape: (152305, 21)


,game_id,play_id,nfl_id,frame_id,x,y,s,a,dir,o,absolute_yardline_number,player_weight,player_height_inches,player_age,is_moving_right,player_position,player_side,player_role,ball_land_x,ball_land_y,num_frames_output
0,2023090700,101,46137,1,51.32,20.69,0.31,0.49,79.43,267.68,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
1,2023090700,101,46137,2,51.35,20.66,0.36,0.74,118.07,268.66,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
2,2023090700,101,46137,3,51.39,20.63,0.44,0.76,130.89,269.78,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
3,2023090700,101,46137,4,51.43,20.61,0.48,0.62,134.50,269.78,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21
4,2023090700,101,46137,5,51.48,20.58,0.54,0.44,129.79,269.06,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21


## Save Feature Engineering Metadata

To improve traceability and reproducibility, we also save metadata describing the engineered dataset.

In [19]:
metadata = {
    "created_at": datetime.now().isoformat(),
    "source_files_used": [f.name for f in selected_files],
    "n_weeks_loaded": n_weeks,
    "n_rows_raw": int(df.shape[0]),
    "n_rows_target": int(target_df.shape[0]),
    "n_rows_model": int(model_df.shape[0]),
    "n_columns_model": int(model_df.shape[1]),
    "engineered_columns": [
        "player_height_inches",
        "player_age",
        "is_moving_right",
    ],
    "target_columns": [
        "ball_land_x",
        "ball_land_y",
        "num_frames_output",
    ],
    "notes": "Baseline feature engineering dataset for target-player rows."
}

metadata_path = PROCESSED_DIR / "baseline_train_v1_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved metadata to:", metadata_path)

Saved metadata to: C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\data\processed\baseline_train_v1_metadata.json


## Summary

In this notebook, we created the first baseline modeling dataset by:

- loading raw weekly training data,
- selecting target-player rows,
- creating simple engineered player features,
- assembling the baseline modeling table,
- validating missing values and target columns,
- and saving both the processed dataset and its metadata.

This processed dataset provides the foundation for the first baseline model training pipeline and future feature improvements.